In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.preprocessing import load_multiple_subjects
from src.data.class_dataset import EEGClassDataset
from src.utils.logging import get_logger
from src.utils.config import (
    LATENT_DIM,
    WINDOW_SIZE,
    WINDOW_STRIDE,
    BATCH_SIZE,
    WANTED_CHANNELS,
    NUM_CHANNELS,
    LR_CLASSIFIER,
    EPOCHS_CLASSIFIER
)
from src.models.classifier import Classifier
from src.training.train_classifier import train_classifier

In [2]:
logger = get_logger("TRTR")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

2026-04-24 20:43:57 | INFO | Using device: cuda


In [4]:
blocks = list(range(1,13))
subj = list(range(1,11))

In [5]:
# eeg = load_subject_frequency(blocks, 12.0, "data/raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
eeg_12 = load_multiple_subjects(subj, blocks, 12.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\timegan2.0\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_12.shape}")
logger.info(f"Mean after norm: {eeg_12.mean(axis=0)}")
logger.info(f"Std after norm: {eeg_12.std(axis=0)}")


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-001_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-002_task-s

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-003_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 526999  =      0.000 ...   526.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-004_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 531999  =      0.000 ...   531.999 secs...


...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-010_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 555999  =      0.000 ...   555.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
2026-04-24 20:45:02 | INFO | EEG shape: (615000, 9)
2026-04-24 20:45:02 | INFO | Mean after norm: [ 6.33134015e-18 -1.25067075e-18 -2.75551939e-18  1.22207574e-17
  1.34598746e-18 -5.41861046e-18 -3.81266834e-19  3.28698225e-18
  8.54673152e-18]
2026-04-24 20:45:02 | INFO | Std after norm: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
eeg_16 = load_multiple_subjects(subj, blocks, 16.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\timegan2.0\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_16.shape}")
logger.info(f"Mean after norm: {eeg_16.mean(axis=0)}")
logger.info(f"Std after norm: {eeg_16.std(axis=0)}")

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-001_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-002_task-s

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-003_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 526999  =      0.000 ...   526.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-001_ses-04_block-004_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 531999  =      0.000 ...   531.999 secs...


...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-010_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 555999  =      0.000 ...   555.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
2026-04-24 20:46:04 | INFO | EEG shape: (615000, 9)
2026-04-24 20:46:04 | INFO | Mean after norm: [ 7.56756897e-19 -5.24530735e-18  1.94677156e-18 -9.70497395e-19
  1.04559541e-18  5.33773567e-18  4.30947058e-18  3.01258566e-18
  3.77223095e-18]
2026-04-24 20:46:04 | INFO | Std after norm: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [7]:
dataset = EEGClassDataset(
    eeg_list=[eeg_12, eeg_16],
    labels=[0,1],
    window_size=WINDOW_SIZE,
    hop_size=WINDOW_STRIDE,
    normalize=False,
)

In [8]:
len(dataset)

9602

In [9]:
train_data, val_data = random_split(dataset, [0.87, 0.13])

In [10]:
train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [11]:
val_dataloader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [12]:
subj = list(range(11,12))

In [13]:
eeg_12_test = load_multiple_subjects(subj, blocks, 12.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\timegan2.0\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_12_test.shape}")
logger.info(f"Mean after norm: {eeg_12_test.mean(axis=0)}")
logger.info(f"Std after norm: {eeg_12_test.std(axis=0)}")

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-001_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 561999  =      0.000 ...   561.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-002_task-s

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-003_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 568999  =      0.000 ...   568.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-004_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 552999  =      0.000 ...   552.999 secs...


...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 627999  =      0.000 ...   627.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
2026-04-24 20:46:10 | INFO | EEG shape: (61500, 9)
2026-04-24 20:46:10 | INFO | Mean after norm: [ 1.69259367e-17  9.93604476e-18 -1.09758634e-17  1.07447926e-17
  2.28760100e-17 -2.84217094e-17 -2.51867181e-17  1.03981864e-18
 -1.55972796e-17]
2026-04-24 20:46:10 | INFO | Std after norm: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [14]:
eeg_16_test = load_multiple_subjects(subj, blocks, 16.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\timegan2.0\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_16_test.shape}")
logger.info(f"Mean after norm: {eeg_16_test.mean(axis=0)}")
logger.info(f"Std after norm: {eeg_16_test.std(axis=0)}")

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-001_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 561999  =      0.000 ...   561.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-002_task-s

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-003_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 568999  =      0.000 ...   568.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-004_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 552999  =      0.000 ...   552.999 secs...


...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\raw\sub-011_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 627999  =      0.000 ...   627.999 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
2026-04-24 20:46:16 | INFO | EEG shape: (61500, 9)
2026-04-24 20:46:16 | INFO | Mean after norm: [-3.11945591e-18 -2.01609280e-17  6.72993729e-18  6.93212425e-19
  1.41530870e-17  6.45554071e-18  5.77677021e-20  2.38002933e-17
 -2.31648485e-17]
2026-04-24 20:46:16 | INFO | Std after norm: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [15]:
test_data = EEGClassDataset(
    eeg_list=[eeg_12_test, eeg_16_test],
    labels=[0,1],
    window_size=WINDOW_SIZE,
    hop_size=WINDOW_STRIDE,
    normalize=False,
)

In [16]:
test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

In [17]:
C = Classifier(9,512,2)

In [18]:
train_classifier(
    C,
    train_dataloader,
    val_dataloader,
    EPOCHS_CLASSIFIER,
    LR_CLASSIFIER,
    device,
    logger
)

2026-04-24 10:34:51 | INFO | Epoch 1 | Train Loss: 0.5726 | Train Acc: 0.6926 | Val Loss: 0.4161 | Val Acc: 0.8558
2026-04-24 10:34:52 | INFO | Epoch 2 | Train Loss: 0.3950 | Train Acc: 0.8335 | Val Loss: 0.2824 | Val Acc: 0.9062
2026-04-24 10:34:54 | INFO | Epoch 3 | Train Loss: 0.3275 | Train Acc: 0.8672 | Val Loss: 0.2422 | Val Acc: 0.9199
2026-04-24 10:34:55 | INFO | Epoch 4 | Train Loss: 0.2950 | Train Acc: 0.8794 | Val Loss: 0.2185 | Val Acc: 0.9351
2026-04-24 10:34:57 | INFO | Epoch 5 | Train Loss: 0.2654 | Train Acc: 0.8948 | Val Loss: 0.2006 | Val Acc: 0.9375
2026-04-24 10:34:58 | INFO | Epoch 6 | Train Loss: 0.2555 | Train Acc: 0.8949 | Val Loss: 0.1801 | Val Acc: 0.9375
2026-04-24 10:35:00 | INFO | Epoch 7 | Train Loss: 0.2410 | Train Acc: 0.9042 | Val Loss: 0.1740 | Val Acc: 0.9407
2026-04-24 10:35:01 | INFO | Epoch 8 | Train Loss: 0.2396 | Train Acc: 0.9047 | Val Loss: 0.1686 | Val Acc: 0.9447
2026-04-24 10:35:02 | INFO | Epoch 9 | Train Loss: 0.2302 | Train Acc: 0.9058 | 

Classifier(
  (temporal): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
  (spatial): Conv2d(8, 16, kernel_size=(9, 1), stride=(1, 1), groups=8, bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (dropout): Dropout(p=0.25, inplace=False)
  (sep): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [19]:
torch.save(C.state_dict(), "TRClassifier2.pt")

In [18]:
C.load_state_dict(torch.load("TRClassifier2.pt", weights_only=True))

<All keys matched successfully>

In [19]:
def evaluate(model, dataloader, device):
    model.to(device)
    model.eval()  # switch to inference mode


    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():  # disable gradients
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)              # (B, num_classes)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())

    accuracy = correct / total

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return accuracy, all_preds, all_labels

In [20]:
accuracy, all_preds, all_labels = evaluate(C, test_dataloader, device)

In [21]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.83      0.95      0.89       477
           1       0.94      0.81      0.87       467

    accuracy                           0.88       944
   macro avg       0.88      0.88      0.88       944
weighted avg       0.88      0.88      0.88       944

